# CMS Open Data Usage — Cleanup & Analysis

Loads the CSV produced by `paper_extractor.py`, cleans it, writes a cleaned CSV, prints summary tables, and shows plots inline.

In [ ]:
# --- Config: set this to your CSV path ---
INPUT_CSV = "cms_dpoa_data_final.csv"

In [ ]:
INPUT_CSV = r"C:\Users\ejren\OneDrive\CMS_DPOA\CMS_DPOA_text_extraction\cms_dpoa_data_final.csv"

In [ ]:
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

## 1. Cleanup helpers

In [ ]:
def normalize_dataset_key(name: str) -> str:
    """Collapse casing/punctuation so the same MC sample across papers groups together."""
    if not isinstance(name, str) or not name.strip():
        return ""
    s = name.lower().strip()
    s = re.sub(r"[\s\-_/]+", "_", s)
    s = re.sub(r"[^\w]", "", s)
    return s


def parse_luminosity_pb(lumi: str) -> float:
    """Parse strings like '31.8/pb', '11.6/fb', '2.3 fb-1' into pb^-1 as a float."""
    if not isinstance(lumi, str) or not lumi.strip():
        return float("nan")
    s = lumi.lower().replace(" ", "").replace("^-1", "").replace("-1", "")
    m = re.search(r"([0-9]*\.?[0-9]+)\s*/?\s*(pb|fb|ab|nb)", s)
    if not m:
        return float("nan")
    value = float(m.group(1))
    unit = m.group(2)
    multipliers = {"nb": 1e-3, "pb": 1.0, "fb": 1e3, "ab": 1e6}
    return value * multipliers[unit]


def flag_suspicious_events(row) -> str:
    """Catch known extraction errors in events_used / events_total."""
    flags = []
    notes = str(row.get("notes", "") or "")
    used = row.get("events_used_num")
    total = row.get("events_total_num")

    if pd.notna(used):
        xs_match = re.search(r"cross[\s-]?section[^\d]*([\d.]+)\s*pb", notes, re.IGNORECASE)
        if xs_match:
            try:
                xs_val = float(xs_match.group(1))
                if abs(xs_val - used) / max(used, 1) < 0.01:
                    flags.append("events_used==cross_section")
            except ValueError:
                pass
        if "weighted events" in notes.lower():
            flags.append("weighted_not_raw")

    if pd.notna(used) and pd.notna(total) and used > total:
        flags.append("used>total")

    return ";".join(flags)

## 2. Load and clean

In [ ]:
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} rows from {INPUT_CSV}")

# Numeric conversions
df["events_total_num"] = pd.to_numeric(df["events_total (from text)"], errors="coerce")
df["events_used_num"] = pd.to_numeric(df["events_used (from text)"], errors="coerce")
df["collision_energy_tev_num"] = pd.to_numeric(df["collision_energy_tev"], errors="coerce")

# Luminosity in pb^-1
df["luminosity_pb"] = df["luminosity"].apply(parse_luminosity_pb)

# Normalized dataset key for cross-paper grouping
df["dataset_key"] = df["dataset_name"].apply(normalize_dataset_key)

# Clean dataset_type
df["dataset_type"] = df["dataset_type"].astype(str).str.strip().str.lower()
df.loc[~df["dataset_type"].isin(["data", "mc", "derived"]), "dataset_type"] = ""

# Fraction of dataset used
df["fraction_used"] = df["events_used_num"] / df["events_total_num"]
df.loc[df["fraction_used"] > 1, "fraction_used"] = float("nan")

# Has official path
df["has_official_path"] = df["official_path"].astype(str).str.strip().ne("")

# Normalize pdf_path slashes
df["pdf_path"] = df["pdf_path"].astype(str).str.replace("\\", "/", regex=False)

# Flag suspicious rows
df["quality_flags"] = df.apply(flag_suspicious_events, axis=1)

# Save cleaned CSV
out_csv = Path(INPUT_CSV).with_name(Path(INPUT_CSV).stem + "_clean.csv")
df.to_csv(out_csv, index=False)
print(f"Wrote cleaned CSV: {out_csv}")
df.head()

## 3. Summary tables

In [ ]:
n_papers = df["paper"].nunique()
n_rows = len(df)
print(f"Papers:           {n_papers}")
print(f"Dataset rows:     {n_rows}")
print(f"Avg datasets/paper: {n_rows / n_papers:.1f}")

In [ ]:
print("Dataset type breakdown:")
print(df["dataset_type"].value_counts(dropna=False))

In [ ]:
print("Papers per year:")
per_year = df.drop_duplicates("paper")["year_published"].value_counts().sort_index()
print(per_year)

In [ ]:
print("Collision energy (TeV):")
print(df["collision_energy_tev_num"].value_counts(dropna=False).sort_index())

In [ ]:
print("Top 15 most-reused datasets (by # of papers citing):")
reuse = (
    df[df["dataset_key"] != ""]
    .groupby("dataset_key")["paper"]
    .nunique()
    .sort_values(ascending=False)
    .head(15)
)
print(reuse)

In [ ]:
print("Official CMS dataset path cited?")
print(df["has_official_path"].value_counts())
print(f"Rate: {df['has_official_path'].mean():.1%}")

In [ ]:
frac = df["fraction_used"].dropna()
print(f"Fraction of dataset processed (rows with both numbers): {len(frac)}")
if len(frac):
    print(f"  median: {frac.median():.3f}")
    print(f"  mean:   {frac.mean():.3f}")
    print(f"  min:    {frac.min():.3f}")
    print(f"  max:    {frac.max():.3f}")

In [ ]:
lumi = df["luminosity_pb"].dropna()
print(f"Rows with luminosity: {len(lumi)}")
if len(lumi):
    print(f"  median: {lumi.median():.2f} pb^-1")
    print(f"  mean:   {lumi.mean():.2f} pb^-1")
    print(f"  range:  {lumi.min():.2f} – {lumi.max():.2f} pb^-1")

In [ ]:
data_rows = df[df["dataset_type"] == "data"]
mc_rows = df[df["dataset_type"] == "mc"]
print("Total events processed across all papers:")
print(f"  real data: {data_rows['events_used_num'].sum():,.0f}")
print(f"  MC:        {mc_rows['events_used_num'].sum():,.0f}")

In [ ]:
flagged = df[df["quality_flags"] != ""]
print(f"Flagged rows needing review: {len(flagged)}")
if len(flagged):
    print(flagged["quality_flags"].value_counts())
    print()
    display(flagged[["paper", "dataset_name", "events_used_num", "quality_flags"]].head(15))

## 4. Plots

In [ ]:
# Papers per year
fig, ax = plt.subplots(figsize=(7, 4))
per_year.plot(kind="bar", ax=ax, color="steelblue")
ax.set_xlabel("Year")
ax.set_ylabel("Papers")
ax.set_title("CMS Open Data papers per year")
plt.tight_layout()
plt.show()

In [ ]:
# Data vs MC
fig, ax = plt.subplots(figsize=(5, 4))
df["dataset_type"].replace("", "unknown").value_counts().plot(
    kind="bar", ax=ax, color=["steelblue", "darkorange", "gray", "lightgray"]
)
ax.set_ylabel("Rows")
ax.set_title("Dataset type breakdown")
plt.tight_layout()
plt.show()

In [ ]:
# Collision energy
energy = df["collision_energy_tev_num"].dropna()
if len(energy):
    fig, ax = plt.subplots(figsize=(5, 4))
    energy.value_counts().sort_index().plot(kind="bar", ax=ax, color="seagreen")
    ax.set_xlabel("Center-of-mass energy (TeV)")
    ax.set_ylabel("Rows")
    ax.set_title("Collision energy distribution")
    plt.tight_layout()
    plt.show()

In [ ]:
# Fraction used histogram
if len(frac):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(frac, bins=20, color="purple", edgecolor="black")
    ax.set_xlabel("events_used / events_total")
    ax.set_ylabel("Rows")
    ax.set_title("Fraction of dataset actually processed")
    plt.tight_layout()
    plt.show()

In [ ]:
# Top reused datasets bar chart
if len(reuse):
    fig, ax = plt.subplots(figsize=(8, 5))
    reuse.sort_values().plot(kind="barh", ax=ax, color="darkred")
    ax.set_xlabel("# of papers citing")
    ax.set_title("Top 15 most-reused datasets")
    plt.tight_layout()
    plt.show()